In [ ]:
#
# Importing all necessary libraries

import numpy as np
import pandas as pd
from pathlib import Path

# %matplotlib inline

In [ ]:
### cell 0 ###

# Alternatively you can use this code to get the dataset
# url = "https://raw.githubusercontent.com/jakevdp/data-CDCbirths/master/births.csv"
birth_data = pd.read_csv(f"{Path(__file__).parent.parent}/input/births.csv")
factor = 50
birth_data = pd.concat([birth_data] * factor)
birth_data.sample(5, random_state=0)

In [ ]:
### cell 1 ###

birth_data.shape, birth_data.info()

In [ ]:
### cell 2 ###

birth_data = birth_data.fillna(0)
birth_data["day"] = birth_data["day"].astype(int)
birth_data.info()

In [ ]:
### cell 3 ###

birth_data.sample(5, random_state=0)

In [ ]:
### cell 4 ###

# Code to add the decade column
birth_data["decade"] = 10 * (birth_data["year"] // 10)
birth_data.head()

In [ ]:
### cell 5 ###

# take a look at male and female births as a function of decade
birth_data.pivot_table("births", index="decade", columns="gender", aggfunc="sum")

In [ ]:
### cell 6 ###

# Code to present male and female mean(births) against decade numericaly
birth_data.pivot_table("births", index="decade", columns="gender", aggfunc="mean")

In [ ]:
### cell 7 ###

years2check = [1988, 1989]

for year in years2check:
    yearly_data = birth_data[birth_data["year"] == year]
    print("Births data for - ", year)
    print(yearly_data)

In [ ]:
### cell 8 ###

# Descriptive summary before removing outliers
birth_data.describe()

In [ ]:
### cell 9 ###

# Assuming 'birth_data' is a cuDF DataFrame
# For example:
# data = np.random.normal(loc=4000, scale=500, size=10000)
# birth_data = cudf.DataFrame({'births': data})

# 1. Your original setup code is fine
#    (Use .to_numpy() to transfer data to the CPU for the numpy function)
quartiles = np.percentile(birth_data["births"].to_numpy(), [25, 50, 75])
mu = quartiles[1]
sig = 0.74 * (quartiles[2] - quartiles[0])

# 2. Pre-calculate the final boundary values
lower_bound = mu - 5 * sig
upper_bound = mu + 5 * sig

# 3. Create the query string using an f-string
#    This injects the *values* of lower_bound and upper_bound into the string
query_string = f"(births > {lower_bound}) & (births < {upper_bound})"

# 4. Use the generated string in the query() method
births = birth_data.query(query_string)

# The rest of your code works as expected
birth_data.shape, births.sample(5, random_state=0)

In [ ]:
### cell 10 ###

# Descriptive summary after removal of outliers
births.describe()

In [ ]:
### cell 11 ###

births.year.unique()

In [ ]:
### cell 12 ###

# create a datetime index from the year, month, day
births.index = pd.to_datetime(
    10000 * births.year + 100 * births.month + births.day, format="%Y%m%d"
)
births.head()

In [ ]:
### cell 13 ###

# Creating another column 'dayofweek' using dayofweek attribute of pandas DatetimeIndex
births["dayofweek"] = births.index.dayofweek
births.head()

In [ ]:
### cell 14 ###

mapping = {
    0: "Monday",
    1: "Tuesady",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}
births["dayofweek"] = births["dayofweek"].map(mapping)

In [ ]:
### cell 15 ###

births.head()

In [ ]:
### cell 16 ###

births_by_date = births.pivot_table("births", [births.index.month, births.index.day])
births_by_date.sample(10, random_state=0)

In [ ]:
### cell 17 ###

births_by_date.shape

In [ ]:
### cell 18 ###

# Creating index like timeseries
# births_by_date.index = [pd.datetime(2020, month, day) for (month, day) in births_by_date.index]
# births_by_date.head()
births_by_date.index = [
    pd.Timestamp(2020, month, day) for (month, day) in births_by_date.index
]
births_by_date.head()